# Day 3 — Memory, RAG and MCP

## Objectives

1. Understand conversation memory.
2. Compare window buffer, summary, and vector memory.
3. Build a small RAG pipeline.
4. Retrieve relevant chunks and answer with citations.
5. Build a small MCP server.
6. Expose two tools and one resource.
7. Connect an MCP client.
8. Discover and invoke MCP tools.

---

## Architecture

User
  ↓
Agent
  ├── Conversation Memory
  ├── RAG
  └── MCP Tools
        ↓
    External capabilities

## 1. Window Buffer Memory

Window buffer memory keeps the most recent conversation messages.

Example:

Message 1
Message 2
Message 3
Message 4
Message 5

If the window size is 3:

Message 3
Message 4
Message 5

Older messages are removed from the active context.

In [1]:
conversation = [
    {"role": "user", "content": "My name is Arun."},
    {"role": "assistant", "content": "Nice to meet you, Arun."},
    {"role": "user", "content": "I am learning agentic AI."},
    {"role": "assistant", "content": "Great! Agentic AI is about systems that can reason and act."},
    {"role": "user", "content": "I am currently learning RAG."},
    {"role": "assistant", "content": "RAG combines retrieval with generation."},
]

window_size = 3

recent_messages = conversation[-window_size:]

for message in recent_messages:
    print(f"{message['role']}: {message['content']}")

assistant: Great! Agentic AI is about systems that can reason and act.
user: I am currently learning RAG.
assistant: RAG combines retrieval with generation.


## 2. Summary Memory

Instead of keeping every previous message, we can maintain a compact summary.

Example:

Original conversation:
- User introduced themselves.
- User is learning agentic AI.
- User is learning RAG.
- User prefers Python.
- User completed an agent-core exercise.

Summary:

"The user is learning agentic AI and RAG using Python and has
completed an agent-core exercise."

The summary can be placed into the model context instead of the
entire conversation.

In [2]:
conversation_summary = """
The user is learning agentic AI and RAG using Python.
The user has completed an agent-core exercise.
"""

print(conversation_summary.strip())

The user is learning agentic AI and RAG using Python.
The user has completed an agent-core exercise.


## 3. Vector Memory

Vector memory stores memories as embeddings.

When a new request arrives:

User request
    ↓
Embedding
    ↓
Similarity search
    ↓
Relevant memories
    ↓
Add them to context

Example memories:

1. User prefers Python.
2. User is building an AI assistant.
3. User uses PostgreSQL.
4. User is learning MCP.

Query:

"What language should I use for this implementation?"

Semantic retrieval can identify:

"User prefers Python."

Unlike a window buffer, vector memory does not depend only on
recency. It retrieves memories based on semantic relevance.

In [3]:
memory_types = {
    "Window Buffer": "Keeps recent messages",
    "Summary": "Compresses conversation into a shorter representation",
    "Vector Memory": "Retrieves semantically relevant memories",
}

for name, description in memory_types.items():
    print(f"{name}: {description}")

Window Buffer: Keeps recent messages
Summary: Compresses conversation into a shorter representation
Vector Memory: Retrieves semantically relevant memories


# Part 2 — RAG Pipeline

RAG = Retrieval-Augmented Generation.

Pipeline:

Document
   ↓
Load
   ↓
Chunk
   ↓
Embed
   ↓
Store
   ↓
Retrieve
   ↓
Relevant chunks
   ↓
LLM
   ↓
Answer + Citations

## RAG Source Document

We will create a small artificial document so that the entire
pipeline is easy to understand and reproducible.

In [4]:
document = """
DocuChat is a RAG-powered application for asking questions about documents.

DocuChat stores document chunks and their embeddings in PostgreSQL
using the pgvector extension.

The retrieval process converts the user question into an embedding
and compares it with stored document embeddings.

The most relevant chunks are then provided to the language model as
context.

The final answer should be grounded in the retrieved document
content and should include citations to the source chunks.
"""

print(document)


DocuChat is a RAG-powered application for asking questions about documents.

DocuChat stores document chunks and their embeddings in PostgreSQL
using the pgvector extension.

The retrieval process converts the user question into an embedding
and compares it with stored document embeddings.

The most relevant chunks are then provided to the language model as
context.

The final answer should be grounded in the retrieved document
content and should include citations to the source chunks.



## Step 1 — Load

Loading means reading source documents into the application.

For this exercise our document is already available as a Python string.
In a real RAG application, the loader could read:

- PDF
- TXT
- Markdown
- DOCX
- HTML
- Database records

In [5]:
def chunk_text(text, chunk_size=250):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)

    return chunks


chunks = chunk_text(document, chunk_size=40)

print(f"Total chunks: {len(chunks)}")

for i, chunk in enumerate(chunks, start=1):
    print(f"\n--- Chunk {i} ---")
    print(chunk)

Total chunks: 2

--- Chunk 1 ---
DocuChat is a RAG-powered application for asking questions about documents. DocuChat stores document chunks and their embeddings in PostgreSQL using the pgvector extension. The retrieval process converts the user question into an embedding and compares it with stored document embeddings.

--- Chunk 2 ---
The most relevant chunks are then provided to the language model as context. The final answer should be grounded in the retrieved document content and should include citations to the source chunks.


## Step 3 — Embed

An embedding converts text into a numerical vector.

Text
 ↓
Embedding model
 ↓
Vector

We will use a small sentence-transformers model for this exercise.

In [6]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = embedding_model.encode(chunks)

print("Number of chunks:", len(chunks))
print("Embedding shape:", chunk_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Number of chunks: 2
Embedding shape: (2, 384)


In [7]:
vector_store = []

for i, (chunk, embedding) in enumerate(
    zip(chunks, chunk_embeddings),
    start=1
):
    vector_store.append({
        "chunk_id": i,
        "text": chunk,
        "embedding": embedding,
        "source": "docuchat_demo.txt",
    })

print("Stored chunks:", len(vector_store))

Stored chunks: 2


## Step 5 — Retrieve

User question:

"Where are the embeddings stored?"

Process:

Question
   ↓
Question embedding
   ↓
Similarity comparison
   ↓
Top-k chunks

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

query = "Where are the embeddings stored?"

query_embedding = embedding_model.encode([query])

scores = cosine_similarity(
    query_embedding,
    [item["embedding"] for item in vector_store]
)[0]

top_k = 2

ranked_indices = scores.argsort()[::-1][:top_k]

retrieved_chunks = []

for index in ranked_indices:
    item = vector_store[index]

    retrieved_chunks.append({
        "chunk_id": item["chunk_id"],
        "text": item["text"],
        "source": item["source"],
        "score": float(scores[index]),
    })

for result in retrieved_chunks:
    print("=" * 60)
    print("Chunk ID:", result["chunk_id"])
    print("Score:", result["score"])
    print("Source:", result["source"])
    print(result["text"])

Chunk ID: 1
Score: 0.24916133284568787
Source: docuchat_demo.txt
DocuChat is a RAG-powered application for asking questions about documents. DocuChat stores document chunks and their embeddings in PostgreSQL using the pgvector extension. The retrieval process converts the user question into an embedding and compares it with stored document embeddings.
Chunk ID: 2
Score: 0.23148801922798157
Source: docuchat_demo.txt
The most relevant chunks are then provided to the language model as context. The final answer should be grounded in the retrieved document content and should include citations to the source chunks.


In [9]:
answer = (
    "The embeddings are stored in PostgreSQL using the pgvector extension."
)

citations = [
    {
        "source": result["source"],
        "chunk_id": result["chunk_id"],
        "score": result["score"],
    }
    for result in retrieved_chunks
]

print("Answer:")
print(answer)

print("\nCitations:")
for citation in citations:
    print(
        f"[{citation['chunk_id']}] "
        f"{citation['source']} "
        f"(score={citation['score']:.3f})"
    )

Answer:
The embeddings are stored in PostgreSQL using the pgvector extension.

Citations:
[1] docuchat_demo.txt (score=0.249)
[2] docuchat_demo.txt (score=0.231)


# MCP Architecture

MCP separates the AI application from external capabilities.

Host / Application
        ↓
MCP Client
        ↓
MCP Server
   ┌────┴────┐
   ↓         ↓
 Tools    Resources

In this exercise:

Tools:
1. add
2. multiply

Resource:
info://project

The client discovers the available capabilities and invokes the
appropriate tool.

## Step 26 — Agent Harness

An agent harness is the runtime around the LLM that controls how the
agent receives context, selects tools, executes actions, handles errors,
and records execution state.

Our Day-2 agent loop was:

Perceive → Reason → Act → Observe → Reason/Finish

                    ┌──────────────────────┐
                    │      User Input      │
                    └──────────┬───────────┘
                               ↓
                    ┌──────────────────────┐
                    │   Context Manager   │
                    │                      │
                    │ history / memory     │
                    │ retrieved context    │
                    └──────────┬───────────┘
                               ↓
                    ┌──────────────────────┐
                    │        LLM           │
                    │      Reasoning       │
                    └──────────┬───────────┘
                               ↓
                    ┌──────────────────────┐
                    │    Tool Registry     │
                    └──────────┬───────────┘
                               ↓
                    ┌──────────────────────┐
                    │   Tool Execution     │
                    └──────────┬───────────┘
                               ↓
                    ┌──────────────────────┐
                    │       Observe        │
                    └──────────┬───────────┘
                               │
                    ┌──────────┴───────────┐
                    │                      │
                  Continue               Finish
                    │                      │
                    └──────→ LLM ←────────┘

## Step 27 — Context-Rot Experiment

Context rot refers to degradation in an agent's ability to correctly
use relevant information as the amount of context increases.

Experiment:

1. Create a known target fact.
2. Add increasing amounts of irrelevant history.
3. Ask the same question.
4. Measure whether the answer remains correct.

History sizes:

- 1K tokens
- 5K tokens
- 10K tokens
- 20K tokens

In [14]:
target_fact = "The project uses PostgreSQL with pgvector for vector storage."

history_sizes = [1000, 5000, 10000, 20000]

for size in history_sizes:
    print(f"History size: {size:,} tokens")
    print(f"Target fact: {target_fact}")
    print("-" * 50)

History size: 1,000 tokens
Target fact: The project uses PostgreSQL with pgvector for vector storage.
--------------------------------------------------
History size: 5,000 tokens
Target fact: The project uses PostgreSQL with pgvector for vector storage.
--------------------------------------------------
History size: 10,000 tokens
Target fact: The project uses PostgreSQL with pgvector for vector storage.
--------------------------------------------------
History size: 20,000 tokens
Target fact: The project uses PostgreSQL with pgvector for vector storage.
--------------------------------------------------


## Step 28 — Chunk Size and Overlap Experiment

Goal:

Measure how chunk size and overlap affect retrieval hit-rate.

We will test:

- Small chunks
- Medium chunks
- Large chunks
- Different overlap values

Dataset:

10 questions with known expected source chunks.

Retrieval hit-rate:

Hit-rate = questions where the expected chunk is retrieved / total questions

In [15]:
chunk_configs = [
    {"chunk_size": 40, "overlap": 0},
    {"chunk_size": 40, "overlap": 10},
    {"chunk_size": 80, "overlap": 10},
    {"chunk_size": 120, "overlap": 20},
]

for config in chunk_configs:
    print(
        f"chunk_size={config['chunk_size']}, "
        f"overlap={config['overlap']}"
    )

chunk_size=40, overlap=0
chunk_size=40, overlap=10
chunk_size=80, overlap=10
chunk_size=120, overlap=20


In [17]:
questions = [
    {
        "question": "What database does DocuChat use?",
        "expected_keyword": "PostgreSQL",
    },
    {
        "question": "What extension is used for vector storage?",
        "expected_keyword": "pgvector",
    },
    {
        "question": "What does the retrieval process convert the user question into?",
        "expected_keyword": "embedding",
    },
    {
        "question": "What does the system compare the question embedding with?",
        "expected_keyword": "embeddings",
    },
    {
        "question": "What are provided to the language model?",
        "expected_keyword": "context",
    },
    {
        "question": "What determines which chunks are selected?",
        "expected_keyword": "relevant",
    },
    {
        "question": "What should the final answer be grounded in?",
        "expected_keyword": "document",
    },
    {
        "question": "What should the final answer include?",
        "expected_keyword": "citations",
    },
    {
        "question": "What type of application is DocuChat?",
        "expected_keyword": "RAG",
    },
    {
        "question": "Where are document chunks and their embeddings stored?",
        "expected_keyword": "PostgreSQL",
    },
]

print(f"Evaluation questions: {len(questions)}")

Evaluation questions: 10


In [18]:
def chunk_text(text, chunk_size=40, overlap=0):
    words = text.split()

    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    chunks = []
    start = 0
    chunk_id = 1

    while start < len(words):
        end = start + chunk_size

        chunk = " ".join(words[start:end])

        chunks.append({
            "chunk_id": chunk_id,
            "text": chunk,
        })

        chunk_id += 1

        if end >= len(words):
            break

        start = end - overlap

    return chunks

In [19]:
test_chunks = chunk_text(
    document,
    chunk_size=40,
    overlap=10,
)

print(f"Number of chunks: {len(test_chunks)}")

for chunk in test_chunks:
    print("\n" + "=" * 60)
    print(f"Chunk {chunk['chunk_id']}")
    print(chunk["text"])

Number of chunks: 3

Chunk 1
DocuChat is a RAG-powered application for asking questions about documents. DocuChat stores document chunks and their embeddings in PostgreSQL using the pgvector extension. The retrieval process converts the user question into an embedding and compares it with stored document embeddings.

Chunk 2
into an embedding and compares it with stored document embeddings. The most relevant chunks are then provided to the language model as context. The final answer should be grounded in the retrieved document content and should include citations to the

Chunk 3
the retrieved document content and should include citations to the source chunks.


In [20]:
def build_vector_store(chunk_size, overlap):
    chunks = chunk_text(
        document,
        chunk_size=chunk_size,
        overlap=overlap,
    )

    texts = [chunk["text"] for chunk in chunks]

    embeddings = embedding_model.encode(texts)

    vector_store = []

    for chunk, embedding in zip(chunks, embeddings):
        vector_store.append({
            "chunk_id": chunk["chunk_id"],
            "text": chunk["text"],
            "embedding": embedding,
        })

    return vector_store

In [21]:
def retrieve(query, vector_store, top_k=2):
    query_embedding = embedding_model.encode([query])

    document_embeddings = [
        item["embedding"]
        for item in vector_store
    ]

    scores = cosine_similarity(
        query_embedding,
        document_embeddings,
    )[0]

    ranked_indices = scores.argsort()[::-1][:top_k]

    results = []

    for index in ranked_indices:
        item = vector_store[index]

        results.append({
            "chunk_id": item["chunk_id"],
            "text": item["text"],
            "score": float(scores[index]),
        })

    return results

In [22]:
def evaluate_configuration(chunk_size, overlap, top_k=2):
    vector_store = build_vector_store(
        chunk_size=chunk_size,
        overlap=overlap,
    )

    hits = 0
    results = []

    for item in questions:
        retrieved = retrieve(
            item["question"],
            vector_store,
            top_k=top_k,
        )

        retrieved_text = " ".join(
            result["text"].lower()
            for result in retrieved
        )

        matched = all(
            keyword.lower() in retrieved_text
            for keyword in item["expected_keywords"]
        )

        if matched:
            hits += 1

        results.append({
            "question": item["question"],
            "hit": matched,
            "retrieved_chunks": [
                result["chunk_id"]
                for result in retrieved
            ],
        })

    hit_rate = hits / len(questions)

    return {
        "chunk_size": chunk_size,
        "overlap": overlap,
        "hits": hits,
        "total": len(questions),
        "hit_rate": hit_rate,
        "details": results,
    }

In [24]:
questions = [
    {
        "question": "What database does DocuChat use?",
        "expected_keywords": ["PostgreSQL"],
    },
    {
        "question": "What extension is used for vector storage?",
        "expected_keywords": ["pgvector"],
    },
    {
        "question": "What does the retrieval process convert the user question into?",
        "expected_keywords": ["embedding"],
    },
    {
        "question": "What does the system compare the question embedding with?",
        "expected_keywords": ["stored", "embeddings"],
    },
    {
        "question": "What is provided to the language model?",
        "expected_keywords": ["context"],
    },
    {
        "question": "What should the final answer be grounded in?",
        "expected_keywords": ["document", "content"],
    },
    {
        "question": "What should the final answer include?",
        "expected_keywords": ["citations"],
    },
    {
        "question": "What type of retrieval does the application use?",
        "expected_keywords": ["semantic", "retrieval"],
    },
    {
        "question": "Why does document chunking divide a large document?",
        "expected_keywords": ["smaller", "pieces"],
    },
    {
        "question": "What is the purpose of chunking?",
        "expected_keywords": ["retrieved", "efficiently"],
    },
]

print(f"Evaluation questions: {len(questions)}")

Evaluation questions: 10


In [25]:
chunk_configs = [
    {"chunk_size": 40, "overlap": 0},
    {"chunk_size": 40, "overlap": 10},
    {"chunk_size": 80, "overlap": 10},
    {"chunk_size": 120, "overlap": 20},
]

experiment_results = []

for config in chunk_configs:
    result = evaluate_configuration(
        chunk_size=config["chunk_size"],
        overlap=config["overlap"],
        top_k=2,
    )

    experiment_results.append(result)

    print(
        f"chunk_size={result['chunk_size']}, "
        f"overlap={result['overlap']} → "
        f"{result['hits']}/{result['total']} hits "
        f"({result['hit_rate']:.1%})"
    )

chunk_size=40, overlap=0 → 7/10 hits (70.0%)
chunk_size=40, overlap=10 → 7/10 hits (70.0%)
chunk_size=80, overlap=10 → 7/10 hits (70.0%)
chunk_size=120, overlap=20 → 7/10 hits (70.0%)


In [26]:
import pandas as pd

experiment_table = pd.DataFrame([
    {
        "Chunk Size": result["chunk_size"],
        "Overlap": result["overlap"],
        "Hits": result["hits"],
        "Questions": result["total"],
        "Hit Rate": f"{result['hit_rate']:.1%}",
    }
    for result in experiment_results
])

experiment_table

,Chunk Size,Overlap,Hits,Questions,Hit Rate
0,40,0,7,10,70.0%
1,40,10,7,10,70.0%
2,80,10,7,10,70.0%
3,120,20,7,10,70.0%


In [27]:
for result in experiment_results:
    print("\n" + "=" * 70)
    print(
        f"Chunk size: {result['chunk_size']} | "
        f"Overlap: {result['overlap']}"
    )

    for detail in result["details"]:
        status = "HIT" if detail["hit"] else "MISS"

        print(
            f"{status:4} | "
            f"Chunks: {detail['retrieved_chunks']} | "
            f"{detail['question']}"
        )


Chunk size: 40 | Overlap: 0
HIT  | Chunks: [1, 2] | What database does DocuChat use?
HIT  | Chunks: [1, 2] | What extension is used for vector storage?
HIT  | Chunks: [1, 2] | What does the retrieval process convert the user question into?
HIT  | Chunks: [1, 2] | What does the system compare the question embedding with?
HIT  | Chunks: [2, 1] | What is provided to the language model?
HIT  | Chunks: [2, 1] | What should the final answer be grounded in?
HIT  | Chunks: [2, 1] | What should the final answer include?
MISS | Chunks: [1, 2] | What type of retrieval does the application use?
MISS | Chunks: [2, 1] | Why does document chunking divide a large document?
MISS | Chunks: [2, 1] | What is the purpose of chunking?

Chunk size: 40 | Overlap: 10
HIT  | Chunks: [1, 3] | What database does DocuChat use?
HIT  | Chunks: [1, 2] | What extension is used for vector storage?
HIT  | Chunks: [1, 2] | What does the retrieval process convert the user question into?
HIT  | Chunks: [2, 1] | What does 

## Step 29 — MCP vs Plain Function Calling

Plain function calling connects an LLM directly to functions controlled
by the application.

MCP provides a standardized protocol for exposing and consuming tools,
resources, and other capabilities.

### Plain Function Calling

LLM
 ↓
Application
 ↓
Python Function
 ↓
Result
 ↓
LLM

### MCP

LLM / Agent
 ↓
MCP Client
 ↓
MCP Server
 ↓
Tool / Resource
 ↓
Result
 ↓
MCP Client
 ↓
Agent

In [28]:
comparison = [
    {
        "Aspect": "Simple local function",
        "Function Calling": "Suitable",
        "MCP": "Usually unnecessary",
    },
    {
        "Aspect": "Application directly controls tool",
        "Function Calling": "Suitable",
        "MCP": "Possible",
    },
    {
        "Aspect": "Tool discovery",
        "Function Calling": "Application-defined",
        "MCP": "Protocol-based",
    },
    {
        "Aspect": "External tool server",
        "Function Calling": "Custom integration",
        "MCP": "Natural fit",
    },
    {
        "Aspect": "Resources",
        "Function Calling": "Not inherent",
        "MCP": "Supported",
    },
    {
        "Aspect": "Interoperability",
        "Function Calling": "Custom implementation",
        "MCP": "Standardized protocol",
    },
]

pd.DataFrame(comparison)

,Aspect,Function Calling,MCP
0,Simple local function,Suitable,Usually unnecessary
1,Application directly controls tool,Suitable,Possible
2,Tool discovery,Application-defined,Protocol-based
3,External tool server,Custom integration,Natural fit
4,Resources,Not inherent,Supported
5,Interoperability,Custom implementation,Standardized protocol


### When should I use plain function calling?

Use plain function calling when:

- The function is local.
- The application owns the function.
- There are only a few tools.
- The integration is simple.
- A separate tool protocol is unnecessary.

### When should I consider MCP?

Consider MCP when:

- Tools need a standardized interface.
- Multiple clients may use the same tools.
- Tool discovery is useful.
- Resources need to be exposed.
- The integration should follow an interoperable protocol.

### Key takeaway

MCP does not make ordinary function calling obsolete.

Plain function calling is a direct application-level mechanism.

MCP provides a standardized protocol boundary between clients and
servers.

## Step 30 — What Should an Agent NOT Remember?

An agent should not automatically store every piece of information
from every conversation.

Memory should be selective.

Two important problems are:

### Privacy

Sensitive information should not automatically become long-term memory.

Examples:

- Passwords
- API keys
- Authentication tokens
- Private credentials
- Sensitive personal information

### Staleness

Information that was correct in the past may become incorrect later.

Examples:

- Temporary plans
- Old project status
- Temporary configuration
- Expired instructions
- Short-term preferences

Before storing information in long-term memory, consider:

1. Is it useful later?
2. Is it safe to store?
3. Is it likely to remain valid?
4. Does the user expect it to be remembered?
5. Can it expire or be updated?

In [29]:
memory_examples = [
    {
        "Information": "Preferred programming language",
        "Remember?": "Usually yes",
        "Reason": "Potentially stable preference",
    },
    {
        "Information": "API key",
        "Remember?": "No",
        "Reason": "Secret credential",
    },
    {
        "Information": "Temporary meeting time",
        "Remember?": "Usually no",
        "Reason": "Can quickly become stale",
    },
    {
        "Information": "Long-term project preference",
        "Remember?": "Potentially yes",
        "Reason": "May be useful in future conversations",
    },
    {
        "Information": "Password",
        "Remember?": "No",
        "Reason": "Sensitive credential",
    },
    {
        "Information": "Current task progress",
        "Remember?": "Temporarily",
        "Reason": "Useful but can become stale",
    },
]

pd.DataFrame(memory_examples)

,Information,Remember?,Reason
0,Preferred programming language,Usually yes,Potentially stable preference
1,API key,No,Secret credential
2,Temporary meeting time,Usually no,Can quickly become stale
3,Long-term project preference,Potentially yes,May be useful in future conversations
4,Password,No,Sensitive credential
5,Current task progress,Temporarily,Useful but can become stale


In [30]:
def memory_decision(
    useful_later,
    sensitive,
    likely_to_change,
    explicitly_requested=False,
):
    if sensitive:
        return "DO NOT STORE"

    if explicitly_requested and useful_later:
        return "CONSIDER STORING SAFELY"

    if useful_later and not likely_to_change:
        return "STORE"

    return "DO NOT STORE"


examples = [
    {
        "name": "Programming language preference",
        "useful_later": True,
        "sensitive": False,
        "likely_to_change": False,
    },
    {
        "name": "API key",
        "useful_later": True,
        "sensitive": True,
        "likely_to_change": False,
    },
    {
        "name": "Temporary meeting time",
        "useful_later": False,
        "sensitive": False,
        "likely_to_change": True,
    },
    {
        "name": "Long-term project preference",
        "useful_later": True,
        "sensitive": False,
        "likely_to_change": False,
    },
]

for item in examples:
    decision = memory_decision(
        useful_later=item["useful_later"],
        sensitive=item["sensitive"],
        likely_to_change=item["likely_to_change"],
    )

    print(f"{item['name']}: {decision}")

Programming language preference: STORE
API key: DO NOT STORE
Temporary meeting time: DO NOT STORE
Long-term project preference: STORE
